In [1]:
!python3 -m pip install faker
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker("pt_BR")
np.random.seed(42)

N_ROWS = 120000

# Estruturas de dados do domínio
marcas_modelos = {
    "Samsung": ["Galaxy S23", "Galaxy A54", "Galaxy Z Flip", "Galaxy M14"],
    "Apple": ["iPhone 13", "iPhone 14", "iPhone 15 Pro", "iPhone SE"],
    "Xiaomi": ["Redmi Note 12", "Poco X5", "Xiaomi 13", "Redmi 10C"],
    "Motorola": ["Moto G84", "Edge 40", "Moto G14", "Razr 40"],
    "Realme": ["Realme 11 Pro+", "Realme C55", "Realme GT 2"],
}

regioes_estados = {
    "Sudeste": ["SP", "RJ", "MG", "ES"],
    "Sul": ["PR", "SC", "RS"],
    "Nordeste": ["BA", "PE", "CE", "MA", "PB", "RN", "AL", "PI", "SE"],
    "Centro-Oeste": ["DF", "GO", "MT", "MS"],
    "Norte": ["AM", "PA", "AM", "RO", "TO", "AC", "AP", "RR"],
}

marcas_list = list(marcas_modelos.keys())
regioes_list = list(regioes_estados.keys())

data = []

for i in range(1, N_ROWS + 1):
    marca = np.random.choice(
        marcas_list, p=[0.35, 0.25, 0.20, 0.15, 0.05]
    )  # Distribuição de market share
    modelo = np.random.choice(marcas_modelos[marca])
    regiao = np.random.choice(
        regioes_list, p=[0.45, 0.20, 0.18, 0.10, 0.07]
    )  # Distribuição regional
    estado = np.random.choice(regioes_estados[regiao])

    preco_base = (
        1200
        if marca in ["Xiaomi", "Motorola", "Realme"]
        else (2200 if marca == "Samsung" else 4500)
    )
    preco = round(np.random.normal(loc=preco_base, scale=400), 2)
    preco = max(preco, 499.00)

    data.append(
        {
            "id_pedido": f"PED-{100000 + i}",
            "data_venda": fake.date_between(
                start_date="-2y", end_date="today"
            ).strftime("%Y-%m-%d"),
            "marca": marca,
            "modelo": modelo,
            "categoria": "Celulares e Smartphones",
            "preco": preco,
            "valor_frete": round(np.random.uniform(15.0, 120.0), 2),
            "regiao": regiao,
            "estado": estado,
            "cidade": fake.city(),
            "avaliacao_cliente": np.random.choice(
                [1, 2, 3, 4, 5], p=[0.05, 0.08, 0.12, 0.35, 0.40]
            ),
            "descricao_produto": f"Smartphone {marca} {modelo} com tela de alta definição, câmera avançada, bateria de longa duração e conectividade 5G.",
        }
    )

df = pd.DataFrame(data)

df.to_csv("vendas_celulares_brasil_120k.csv", index=False)
print(f"Dataset gerado com sucesso! Total de linhas: {len(df)}")

Dataset gerado com sucesso! Total de linhas: 120000


In [3]:
# ============================================================
# RELATÓRIO DE QUALIDADE DE DADOS — Vendas Celulares Brasil
# Great Expectations 1.x
# ============================================================

!python3 -m pip install dataframe_image
!python3 -m pip install great_expectations

import dataframe_image as dfi
from datetime import datetime
import great_expectations as gx
import pandas as pd

df = pd.read_csv("vendas_celulares_brasil_120k.csv")
context = gx.get_context()

data_source = context.data_sources.add_pandas("vendas_celulares")
data_asset = data_source.add_dataframe_asset(name="vendas_120k")
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "batch_vendas"
)
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# ── 3. DEFINIR SUITE DE EXPECTATIONS
suite = gx.ExpectationSuite(name="vendas_qualidade_suite")
context.suites.add(suite)

# ── DIMENSÃO: COMPLETENESS
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="id_pedido",
        meta={
            "dimension": "Completeness",
            "description": "ID do pedido não pode ser nulo",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="preco",
        meta={
            "dimension": "Completeness",
            "description": "Preço não pode ser nulo",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="regiao",
        meta={
            "dimension": "Completeness",
            "description": "Região não pode ser nula",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="data_venda",
        meta={
            "dimension": "Completeness",
            "description": "Data da venda não pode ser nula",
        },
    )
)

# ── DIMENSÃO: UNIQUENESS
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="id_pedido",
        meta={
            "dimension": "Uniqueness",
            "description": "Cada pedido deve ter ID único",
        },
    )
)

# ── DIMENSÃO: VALIDITY (Domínio/Formato)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="preco",
        min_value=1.0,
        max_value=100000.0,
        meta={
            "dimension": "Validity",
            "description": "Preço entre R$1 e R$100.000",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="valor_frete",
        min_value=0.0,
        max_value=5000.0,
        meta={
            "dimension": "Validity",
            "description": "Frete entre R$0 e R$5.000",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="regiao",
        value_set=["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"],
        meta={
            "dimension": "Validity",
            "description": "Região deve ser uma das 5 regiões BR",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="data_venda",
        regex=r"\d{4}-\d{2}-\d{2}",
        meta={
            "dimension": "Validity",
            "description": "Data em formato YYYY-MM-DD",
        },
    )
)

# ── DIMENSÃO: ACCURACY
suite.add_expectation(
    gx.expectations.ExpectColumnMeanToBeBetween(
        column="preco",
        min_value=500.0,
        max_value=8000.0,
        meta={
            "dimension": "Accuracy",
            "description": "Preço médio plausível para celulares",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=100000,
        max_value=200000,
        meta={
            "dimension": "Accuracy",
            "description": "Volume esperado: 100k-200k linhas",
        },
    )
)

# ── DIMENSÃO: CONSISTENCY
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="preco",
        type_="float",
        meta={
            "dimension": "Consistency",
            "description": "Preço deve ser numérico (float)",
        },
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column="valor_frete",
        type_="float",
        meta={
            "dimension": "Consistency",
            "description": "Frete deve ser numérico (float)",
        },
    )
)

# ── 4. VALIDAR
validator = context.get_validator(
    batch=batch, expectation_suite_name="vendas_qualidade_suite"
)
validation_result = validator.validate()

# ── 5. RELATÓRIO ESTRUTURADO
print("=" * 70)
print(f"  RELATÓRIO DE QUALIDADE DE DADOS")
print(f"  Dataset: vendas_celulares_brasil_120k.csv")
print(f"  Data: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Linhas: {len(df):,} | Colunas: {len(df.columns)}")
print("=" * 70)

rows = []
for r in validation_result["results"]:
  cfg = r["expectation_config"]
  rows.append({
      "Dimensão": cfg.get("meta", {}).get("dimension", "—"),
      "Coluna": cfg["kwargs"].get("column", "Tabela"),
      "Regra": cfg["type"].replace("expect_", "").replace("_", " "),
      "Status": "✅ PASS" if r["success"] else "❌ FAIL",
      "Falhas": (
          r["result"].get("unexpected_count", 0) if not r["success"] else 0
      ),
      "% Falhas": (
          f"{r['result'].get('unexpected_percent', 0):.2f}%"
          if not r["success"]
          else "0.00%"
      ),
  })

report_df = pd.DataFrame(rows)
print(report_df.to_string(index=False))

# ── 6. RESUMO
print("\n" + "=" * 70)
print("  RESUMO POR DIMENSÃO")
print("=" * 70)
summary = report_df.groupby("Dimensão")["Status"].apply(
    lambda x: f"{(x == '✅ PASS').sum()} pass / {(x == '❌ FAIL').sum()} fail"
)
print(summary.to_string())

overall = validation_result["statistics"]["success_percent"]
print(f"\n TAXA DE CONFORMIDADE: {overall:.1f}%")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.2 MB/s eta 0:00:00


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpd9ekv37a' for ephemeral docs site


Calculating Metrics:   0%|          | 0/33 [00:00<?, ?it/s]

  RELATÓRIO DE QUALIDADE DE DADOS
  Dataset: vendas_celulares_brasil_120k.csv
  Data: 2026-10-03 19:23
  Linhas: 120,000 | Colunas: 12
    Dimensão      Coluna                         Regra Status  Falhas % Falhas
Completeness   id_pedido  column values to not be null ✅ PASS       0    0.00%
  Uniqueness   id_pedido    column values to be unique ✅ PASS       0    0.00%
Completeness       preco  column values to not be null ✅ PASS       0    0.00%
    Validity       preco   column values to be between ✅ PASS       0    0.00%
    Accuracy       preco     column mean to be between ✅ PASS       0    0.00%
 Consistency       preco   column values to be of type ✅ PASS       0    0.00%
Completeness      regiao  column values to not be null ✅ PASS       0    0.00%
    Validity      regiao    column values to be in set ✅ PASS       0    0.00%
Completeness  data_venda  column values to not be null ✅ PASS       0    0.00%
    Validity  data_venda  column values to match regex ✅ PASS       0    0.

In [4]:
# ============================================================
# COMMON DATA MODEL — E-commerce Celulares
# ============================================================

CDM_SCHEMA = {
    "entity": "orders",
    "version": "1.0.0",
    "description": "Modelo padrão para pedidos de e-commerce de celulares",
    "columns": {
        # ── Identificadores ──
        "order_id": {
            "type": "string",
            "nullable": False,
            "unique": True,
            "description": "Identificador único do pedido",
        },
        "product_id": {
            "type": "string",
            "nullable": False,
            "unique": False,
            "description": "Modelo do produto",
        },
        # ── Dimensões Temporais ──
        "order_date": {
            "type": "date",
            "nullable": False,
            "format": "YYYY-MM-DD",
            "description": "Data do pedido",
        },
        # ── Fatos / Medidas ──
        "unit_price": {
            "type": "decimal(10,2)",
            "nullable": False,
            "currency": "BRL",
            "description": "Preço unitário do produto",
        },
        "shipping_cost": {
            "type": "decimal(10,2)",
            "nullable": False,
            "currency": "BRL",
            "description": "Custo de frete",
        },
        "total_amount": {
            "type": "decimal(10,2)",
            "nullable": False,
            "currency": "BRL",
            "description": "Valor total (unit_price + shipping_cost)",
        },
        # ── Dimensões Geográficas ──
        "region": {
            "type": "string",
            "nullable": False,
            "allowed_values": [
                "Sudeste",
                "Sul",
                "Nordeste",
                "Centro-Oeste",
                "Norte",
            ],
            "description": "Região do Brasil",
        },
        "state": {
            "type": "string",
            "nullable": False,
            "length": 2,
            "description": "UF (sigla de 2 letras)",
        },
        "city": {"type": "string", "nullable": False, "description": "Cidade"},
        # ── Status e Avaliação ──
        "order_status": {
            "type": "string",
            "nullable": False,
            "description": "Status atual do pedido",
        },
        "rating": {
            "type": "integer",
            "nullable": True,
            "description": "Avaliação do cliente (1-5)",
        },
    },
}

# ── Colunas originais → CDM
COLUMN_MAPPING = {
    "id_pedido": "order_id",
    "data_venda": "order_date",
    "preco": "unit_price",
    "valor_frete": "shipping_cost",
    "regiao": "region",
    "estado": "state",
    "cidade": "city",
    "modelo": "product_id",
    "avaliacao_cliente": "rating",
}


# ── Aplicar CDM
def apply_cdm(df_raw: pd.DataFrame) -> pd.DataFrame:
  """Renomeia colunas e aplica o CDM."""
  df_cdm = df_raw.rename(columns=COLUMN_MAPPING).copy()

  if "unit_price" in df_cdm.columns and "shipping_cost" in df_cdm.columns:
    df_cdm["total_amount"] = (
        df_cdm["unit_price"] + df_cdm["shipping_cost"]
    ).round(2)

  if "order_status" not in df_cdm.columns:
    df_cdm["order_status"] = "delivered"

  df_cdm["unit_price"] = df_cdm["unit_price"].astype("float64")
  df_cdm["shipping_cost"] = df_cdm["shipping_cost"].astype("float64")
  df_cdm["total_amount"] = df_cdm["total_amount"].astype("float64")
  df_cdm["order_date"] = pd.to_datetime(df_cdm["order_date"]).dt.date

  return df_cdm


df_cdm = apply_cdm(df)

print("\n✅ CDM aplicado com sucesso!")
print(f"Colunas finais no CDM: {df_cdm.columns.tolist()}")
print("\nPrimeiros 3 registros transformados:")
print(df_cdm.head(3))

df_cdm.to_parquet("vendas_celulares_cdm.parquet", index=False)
df_cdm.to_csv("vendas_celulares_cdm.csv", index=False)


✅ CDM aplicado com sucesso!
Colunas finais no CDM: ['order_id', 'order_date', 'marca', 'product_id', 'categoria', 'unit_price', 'shipping_cost', 'region', 'state', 'city', 'rating', 'descricao_produto', 'total_amount', 'order_status']

Primeiros 3 registros transformados:
     order_id  order_date     marca product_id                categoria  \
0  PED-100001  2025-06-10     Apple  iPhone 13  Celulares e Smartphones   
1  PED-100002  2025-03-16  Motorola    Razr 40  Celulares e Smartphones   
2  PED-100003  2026-09-08  Motorola    Edge 40  Celulares e Smartphones   

   unit_price  shipping_cost   region state                city  rating  \
0     4055.25          31.38  Sudeste    ES     Santos do Oeste       2   
1     1327.56          17.16  Sudeste    MG              Novais       5   
2      971.45          70.10  Sudeste    ES  das Neves de Jesus       4   

                                   descricao_produto  total_amount  \
0  Smartphone Apple iPhone 13 com tela de alta de...  